# Airbnb Agentic RAG — Evaluation Steps Explained

This notebook walks through **every step** of the evaluation pipeline — exactly what
`eval/evaluate.py` does, broken into small, inspectable pieces.

### What you'll learn
| Step | What happens |
|------|--------------|
| 1 | Load and inspect `eval/dataset.json` |
| 2 | Connect to the API and verify it is the correct server |
| 3 | Call `/rag` for one query and see the raw response |
| 4 | Score that response with `keyword_hit_rate` |
| 5 | Call `/ask` for one query and see tool_calls in detail |
| 6 | Score `/ask` with `filter_accuracy` |
| 7 | Use Gemini as LLM judge — build prompt, call model, parse score |
| 8 | Run a mini evaluation (5 queries) |
| 9 | Aggregate results into a summary |
| 10 | Visualise results in a DataFrame |
| 11 | Build the HTML report string |
| 12 | Upload everything to GCS |

### Run from the project root
```bash
cd AirbnbAgenticRAG/           # project root
jupyter notebook notebooks/evaluation_steps_explained_notebook.ipynb
```

### Pre-requisites
- Cloud Run API is deployed and healthy (see `ragrun.html`)
- `gcloud auth application-default login` has been run
- `pip install -r requirements.txt` (or `-r requirements-api.txt`) done

---
## Step 0 — Setup: add project root to Python path

The notebook lives in `notebooks/` but all modules (`config`, `eval/`) are one level up.
We add the project root so imports work exactly the same as when running
`python eval/evaluate.py` from the root directory.

In [2]:
import sys
from pathlib import Path

# ---------------------------------------------------------------------------
# Resolve the project root (one directory above 'notebooks/')
# Path(__file__) is unreliable in notebooks, so we use Path.cwd() instead.
# If you open the notebook from the project root, cwd() == project root.
# If you open from notebooks/, we go one level up.
# ---------------------------------------------------------------------------
HERE = Path.cwd()
ROOT = HERE.parent if HERE.name == "notebooks" else HERE

# Insert at position 0 so our modules take priority over any installed packages
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print(f"Project root : {ROOT}")
print(f"Python path[0]: {sys.path[0]}")

Project root : /Users/kamaldhungana/Documents/Coding/GeminiCours101/GCP-RAG-Design/RAG-VertexAI/AirbnbAgenticRAG
Python path[0]: /Users/kamaldhungana/Documents/Coding/GeminiCours101/GCP-RAG-Design/RAG-VertexAI/AirbnbAgenticRAG


In [3]:
# ---------------------------------------------------------------------------
# Standard library imports — no GCP dependencies yet
# ---------------------------------------------------------------------------
import json
import time
import re
from datetime import datetime, timezone
from pprint import pprint

import requests

# ---------------------------------------------------------------------------
# Our project modules
# ---------------------------------------------------------------------------
import config                          # single source of truth for all settings

from eval.metrics import (
    keyword_hit_rate,      # lexical overlap: are expected words in the answer?
    filter_accuracy,       # did /ask extract the right structured filters?
    build_judge_prompt,    # builds the Gemini-as-judge prompt string
    parse_judge_score,     # parses the integer score from Gemini's response
    aggregate,             # computes summary stats across all results
)
from eval.report import build_html_report  # renders HTML report as a string

print("Imports OK")

Imports OK


---
## Step 1 — Load and explore the evaluation dataset

The dataset is a **static JSON file** checked into the repo at `eval/dataset.json`.
It never changes at runtime — the evaluator only reads it, never writes to it.

### Dataset design choices
- **25 queries** — small enough to run in < 10 minutes, diverse enough to be meaningful
- **5 categories** — each tests a different RAG capability
- **`endpoint` field** — tells the runner which API endpoint to call for each query
  - `"both"` → call `/rag` AND `/ask`
  - `"rag"` → only `/rag` (semantic search, no agent)
  - `"ask"` → only `/ask` (agentic with structured filter extraction)

In [4]:
# ---------------------------------------------------------------------------
# Load the dataset from the path defined in config.py
# config.EVAL_DATASET = "eval/dataset.json"  (relative to project root)
# ---------------------------------------------------------------------------
dataset_path = ROOT / config.EVAL_DATASET
print(f"Loading dataset from: {dataset_path}")

with open(dataset_path) as f:
    dataset = json.load(f)

queries = dataset["queries"]

print(f"\nDataset version   : {dataset['version']}")
print(f"Total queries     : {len(queries)}")
print(f"\nCategory descriptions:")
for cat, desc in dataset["categories"].items():
    print(f"  {cat:15s} — {desc}")

Loading dataset from: /Users/kamaldhungana/Documents/Coding/GeminiCours101/GCP-RAG-Design/RAG-VertexAI/AirbnbAgenticRAG/eval/dataset.json

Dataset version   : 1.0
Total queries     : 25

Category descriptions:
  semantic        — Pure natural-language queries, no hard constraints
  price           — Price-bounded queries that require correct filter extraction
  room_type       — Room-type constrained queries
  multi_filter    — Combined price + room + neighbourhood constraints
  neighbourhood   — Location-specific queries


In [5]:
dataset

{'version': '1.0',
 'description': 'Airbnb Agentic RAG evaluation dataset — 25 labeled queries across 5 categories',
 'created': '2026-03-18',
 'categories': {'semantic': 'Pure natural-language queries, no hard constraints',
  'price': 'Price-bounded queries that require correct filter extraction',
  'room_type': 'Room-type constrained queries',
  'multi_filter': 'Combined price + room + neighbourhood constraints',
  'neighbourhood': 'Location-specific queries'},
 'queries': [{'id': 'q001',
   'category': 'semantic',
   'endpoint': 'both',
   'query': 'cozy place to relax with a great view',
   'expected_filters': {},
   'expected_answer_keywords': ['cozy', 'view', 'relax', 'Austin'],
   'min_results_expected': 1,
   'notes': 'Pure semantic — no structured filters expected'},
  {'id': 'q002',
   'category': 'semantic',
   'endpoint': 'both',
   'query': 'stylish modern apartment close to restaurants and nightlife',
   'expected_filters': {},
   'expected_answer_keywords': ['apartment',

In [12]:
# ---------------------------------------------------------------------------
# Show a summary of all 25 queries in a readable table
# ---------------------------------------------------------------------------
from collections import Counter

print(f"{'ID':6} {'Category':15} {'Endpoint':8} {'Query (truncated)'}")
print("-" * 80)
for q in queries:
    print(f"{q['id']:6} {q['category']:15} {q['endpoint']:8} {q['query'][:50]}")

# Show count per category and endpoint type
print("\nCount by category:")
for cat, cnt in Counter(q["category"] for q in queries).items():
    print(f"  {cat:15s} : {cnt}")

print("\nCount by endpoint:")
for ep, cnt in Counter(q["endpoint"] for q in queries).items():
    print(f"  {ep:8s} : {cnt}")

ID     Category        Endpoint Query (truncated)
--------------------------------------------------------------------------------
q001   semantic        both     cozy place to relax with a great view
q002   semantic        both     stylish modern apartment close to restaurants and 
q003   semantic        both     quiet retreat away from the city noise, good for w
q004   semantic        both     romantic getaway with a pool
q005   semantic        both     pet-friendly home with a yard
q006   price           ask      find me a private room under $60 a night
q007   price           ask      entire apartment between $80 and $150 per night
q008   price           ask      cheap place to stay, max $50 per night
q009   price           ask      luxury stay over $300 a night with great ratings
q010   price           ask      affordable entire home for a week, budget around $
q011   room_type       ask      I need a private room in someone's home, not the w
q012   room_type       ask      I want 

In [13]:
# ---------------------------------------------------------------------------
# Inspect one query in full detail — q016 is the most complex (4 filters)
# ---------------------------------------------------------------------------
q016 = next(q for q in queries if q["id"] == "q016")

print("Full query definition for q016 (multi_filter category):")
print("-" * 60)
pprint(q016)
print()
print("Key fields explained:")
print(f"  query              : The natural language question sent to the API")
print(f"  endpoint           : 'ask' → only /ask is evaluated (has hard filters)")
print(f"  expected_filters   : What the Gemini agent SHOULD extract from the query")
print(f"  expected_answer_keywords : Words that should appear in the answer text")
print(f"  min_results_expected     : At least N listings must be returned")
print(f"  notes              : Human annotation for why this query is interesting")

Full query definition for q016 (multi_filter category):
------------------------------------------------------------
{'category': 'multi_filter',
 'endpoint': 'ask',
 'expected_answer_keywords': ['bedroom', '$120', 'downtown'],
 'expected_filters': {'max_price': 120,
                      'min_bedrooms': 2,
                      'neighbourhood': 'downtown',
                      'room_type': 'Entire home/apt'},
 'id': 'q016',
 'min_results_expected': 1,
 'notes': '4 simultaneous filters — hardest for agent',
 'query': '2-bedroom entire home under $120 near downtown Austin'}

Key fields explained:
  query              : The natural language question sent to the API
  endpoint           : 'ask' → only /ask is evaluated (has hard filters)
  expected_filters   : What the Gemini agent SHOULD extract from the query
  expected_answer_keywords : Words that should appear in the answer text
  min_results_expected     : At least N listings must be returned
  notes              : Human annotation 

---
## Step 2 — Connect to the API and verify it is the right server

> **Critical:** Always point at Cloud Run — not localhost.  
> Running against a wrong server produces `results=0`, `judge=None`, and a 0.2s total time.
> The health endpoint must return `collection` and `cache` keys — that is our API.

Set `API_URL` to your Cloud Run URL:

In [14]:
#os.environ.get("RAG_API_URL", "").rstrip("/")

In [15]:
# ---------------------------------------------------------------------------
# Set the API URL — three options:
#
#   Option A (recommended): use the RAG_API_URL env var
#     export RAG_API_URL=$(gcloud run services describe airbnb-rag-api \
#       --region=us-central1 --project=deve-487713 --format="value(status.url)")
#
#   Option B: hardcode directly in this cell
#     API_URL = "https://airbnb-rag-api-xxxxxxxxxx-uc.a.run.app"
#
#   Option C: config.py reads RAG_API_URL env var and defaults to localhost
#     API_URL = config.API_BASE_URL
# ---------------------------------------------------------------------------
import os

API_URL = os.environ.get("RAG_API_URL", "").rstrip("/")
if API_URL=='':
    API_URL="https://airbnb-rag-api-zdpzfpmtka-uc.a.run.app"

if not API_URL:
    # Uncomment and set your Cloud Run URL here:
    # API_URL = "https://airbnb-rag-api-xxxxxxxxxx-uc.a.run.app"
    raise ValueError(
        "RAG_API_URL is not set.\n"
        "Run in terminal:\n"
        "  export RAG_API_URL=$(gcloud run services describe airbnb-rag-api "
        "--region=us-central1 --project=deve-487713 --format='value(status.url)')\n"
        "Then restart this notebook kernel."
    )

print(f"API URL: {API_URL}")

API URL: https://airbnb-rag-api-zdpzfpmtka-uc.a.run.app


In [16]:
# ---------------------------------------------------------------------------
# Health check — verify this is OUR API (not some other service on port 8000)
#
# Our /health endpoint returns:
#   { "status": "ok",
#     "collection": "projects/.../collections/airbnb-listings-collection",
#     "cache": { "status": "connected", "host": "10.x.x.x", ... } }
#
# A WRONG server returns something like { "instantiated_at": "..." }  ← no 'collection' key
# ---------------------------------------------------------------------------
resp = requests.get(f"{API_URL}/health", timeout=10)
resp.raise_for_status()
health = resp.json()

print("Health check response:")
pprint(health)

# Validate it is OUR API
assert "collection" in health, (
    "This does NOT look like our API.\n"
    "Expected key 'collection' in /health response.\n"
    "You may be hitting a different service. Set RAG_API_URL to the Cloud Run URL."
)
assert "cache" in health, "Expected key 'cache' in /health response."

print("\n✓  Confirmed: correct API")
print(f"   Collection : {health['collection']}")
print(f"   Cache      : {health['cache'].get('status', 'unknown')} "
      f"on {health['cache'].get('host', '?')}")

Health check response:
{'cache': {'ask_ttl_s': 1800,
           'backend': 'memorystore_redis',
           'db': 0,
           'enabled': True,
           'hits': 5,
           'host': '10.36.243.131',
           'misses': 96,
           'port': 6379,
           'rag_ttl_s': 3600,
           'redis_version': '7.0.15',
           'status': 'connected',
           'total_keys': 10,
           'used_memory_mb': 3.94},
 'collection': 'projects/deve-487713/locations/us-central1/collections/airbnb-listings-collection',
 'status': 'ok'}

✓  Confirmed: correct API
   Collection : projects/deve-487713/locations/us-central1/collections/airbnb-listings-collection
   Cache      : connected on 10.36.243.131


---
## Step 3 — Call `/rag` for a single query

The `/rag` endpoint does **Simple RAG**:
1. Embed the query → 768-dim vector (Vertex AI `text-embedding-005`)
2. ANN search in VS2.0 collection → top-K DataObject IDs + scores
3. Fetch each DataObject metadata (listing details)
4. Pass listings + query to Gemini → natural language answer

No structured filter extraction — purely semantic similarity.

In [17]:
# ---------------------------------------------------------------------------
# Pick query q001 — a pure semantic query (no hard filters)
# ---------------------------------------------------------------------------
q001 = next(q for q in queries if q["id"] == "q001")
print(f"Query  : {q001['query']}")
print(f"Category: {q001['category']}")
print(f"Expected keywords: {q001['expected_answer_keywords']}")
print(f"Min results expected: {q001['min_results_expected']}")

Query  : cozy place to relax with a great view
Category: semantic
Expected keywords: ['cozy', 'view', 'relax', 'Austin']
Min results expected: 1


In [18]:
# ---------------------------------------------------------------------------
# Call POST /rag
#
# Request body:
#   query : str  — the natural language query
#   top_k : int  — how many listings to retrieve from VS2.0
#
# We time the call with perf_counter (high-resolution monotonic clock)
# ---------------------------------------------------------------------------
t0 = time.perf_counter()

rag_response = requests.post(
    f"{API_URL}/rag",
    json={
        "query": q001["query"],
        "top_k": config.EVAL_TOP_K,   # = 10 from config.py
    },
    timeout=60,
)
rag_response.raise_for_status()

latency_ms = (time.perf_counter() - t0) * 1000
rag_data = rag_response.json()

print(f"HTTP status : {rag_response.status_code}")
print(f"Latency     : {latency_ms:.0f} ms")
print(f"\nResponse keys: {list(rag_data.keys())}")

HTTP status : 200
Latency     : 133 ms

Response keys: ['answer', 'sources', 'model', 'collection']


In [19]:
# ---------------------------------------------------------------------------
# Inspect the /rag response structure
#
# The response has two main parts:
#   'answer'  — the Gemini-generated natural language answer
#   'sources' — the list of retrieved Airbnb listings (raw DataObject metadata)
# ---------------------------------------------------------------------------
answer  = rag_data.get("answer", "")
sources = rag_data.get("sources", [])

print(f"Number of listings retrieved: {len(sources)}")
print(f"\n--- Gemini Answer ---")
print(answer)
print(f"\n--- First source (listing) ---")
if sources:
    pprint(sources[0])   # shows all metadata fields for the first listing

Number of listings retrieved: 10

--- Gemini Answer ---
Here are some cozy places to relax with a great view:

*   **Cozy cottage in the trees**
    Location: 78733
    Price: $174/night
    Size: Accommodates 2 guests, 1 bedroom
    URL: https://www.airbnb.com/rooms/16644078

*   **Tranquil Nest With A View**
    Location: 78741
    Price: $81/night
    Size: Accommodates 4 guests, 2 bedrooms
    URL: https://www.airbnb.com/rooms/51169203

*   **Remodeled Top Floor Condo w/Great Views**
    Location: 78730
    Price: $86/night
    Size: Accommodates 2 guests, 2 bedrooms
    Rating: 4.91 ★
    URL: https://www.airbnb.com/rooms/13436718

--- First source (listing) ---
{'accommodates': 2.0,
 'host_name': 'Paul',
 'id': 'd088b0bc5cc71ab54c1afac22bda49e7b7b1cf2e',
 'instant_bookable': 'false',
 'listing_url': 'https://www.airbnb.com/rooms/52368571',
 'name': 'Relaxing Spot',
 'neighbourhood': '78758',
 'price': 54.0,
 'rank': 1,
 'rating': 4.96,
 'room_type': 'Private room',
 'score': 0.65

---
## Step 4 — Score a `/rag` response with `keyword_hit_rate`

**`keyword_hit_rate`** measures whether the expected keywords appear in the answer.

Formula:  
```
khr = (number of expected keywords found in answer) / (total expected keywords)
```

- Returns `1.0` if `expected_answer_keywords` is empty (no expectation → auto-pass)
- Matching is **case-insensitive** and substring-based
- Limitation: it only checks presence, not context — a keyword can appear in a
  negative sentence and still count as a hit

In [20]:
# ---------------------------------------------------------------------------
# Source code of keyword_hit_rate (from eval/metrics.py)
# ---------------------------------------------------------------------------
import inspect
print(inspect.getsource(keyword_hit_rate))

def keyword_hit_rate(answer: str, expected_keywords: list[str]) -> float:
    """
    Fraction of expected_keywords that appear (case-insensitive) in the answer.

    Returns 1.0 if expected_keywords is empty (no constraint → auto-pass).
    """
    if not expected_keywords:
        return 1.0
    answer_lower = answer.lower()
    hits = sum(1 for kw in expected_keywords if kw.lower() in answer_lower)
    return round(hits / len(expected_keywords), 4)



In [21]:
# ---------------------------------------------------------------------------
# Compute keyword_hit_rate for q001
# ---------------------------------------------------------------------------
expected_keywords = q001["expected_answer_keywords"]
print(f"Expected keywords : {expected_keywords}")
print(f"Answer (first 300 chars):\n  {answer[:300]}")
print()

# Manual step-by-step to understand the calculation:
answer_lower = answer.lower()
for kw in expected_keywords:
    found = kw.lower() in answer_lower
    print(f"  '{kw}' → {'✓ FOUND' if found else '✗ MISSING'}")

# Using the actual metric function:
khr = keyword_hit_rate(answer, expected_keywords)
print(f"\nkeyword_hit_rate = {khr:.4f}  ({khr*100:.0f}% of keywords found)")

Expected keywords : ['cozy', 'view', 'relax', 'Austin']
Answer (first 300 chars):
  Here are some cozy places to relax with a great view:

*   **Cozy cottage in the trees**
    Location: 78733
    Price: $174/night
    Size: Accommodates 2 guests, 1 bedroom
    URL: https://www.airbnb.com/rooms/16644078

*   **Tranquil Nest With A View**
    Location: 78741
    Price: $81/night
   

  'cozy' → ✓ FOUND
  'view' → ✓ FOUND
  'relax' → ✓ FOUND
  'Austin' → ✗ MISSING

keyword_hit_rate = 0.7500  (75% of keywords found)


In [22]:
# ---------------------------------------------------------------------------
# has_results — did the response meet the minimum result count?
# ---------------------------------------------------------------------------
retrieved_count   = len(sources)
min_expected      = q001["min_results_expected"]
has_results       = retrieved_count >= min_expected

print(f"Retrieved count   : {retrieved_count}")
print(f"Min expected      : {min_expected}")
print(f"has_results       : {has_results}")

# Build the final result dict for this query (same structure as evaluate.py)
rag_result_q001 = {
    "id":               q001["id"],
    "category":         q001["category"],
    "query":            q001["query"],
    "endpoint":         "rag",
    "answer":           answer[:500],
    "error":            None,
    "has_results":      has_results,
    "retrieved_count":  retrieved_count,
    "latency_ms":       round(latency_ms, 1),
    "keyword_hit_rate": khr,
    "answer_relevance": None,   # filled in Step 7 (LLM judge)
    "filter_accuracy":  None,   # N/A for /rag
}
pprint(rag_result_q001)

Retrieved count   : 10
Min expected      : 1
has_results       : True
{'answer': 'Here are some cozy places to relax with a great view:\n'
           '\n'
           '*   **Cozy cottage in the trees**\n'
           '    Location: 78733\n'
           '    Price: $174/night\n'
           '    Size: Accommodates 2 guests, 1 bedroom\n'
           '    URL: https://www.airbnb.com/rooms/16644078\n'
           '\n'
           '*   **Tranquil Nest With A View**\n'
           '    Location: 78741\n'
           '    Price: $81/night\n'
           '    Size: Accommodates 4 guests, 2 bedrooms\n'
           '    URL: https://www.airbnb.com/rooms/51169203\n'
           '\n'
           '*   **Remodeled Top Floor Condo w/Great Views**\n'
           '    Location: 78730\n'
           '    Price: $86/night\n'
           '    Size: Accommodates',
 'answer_relevance': None,
 'category': 'semantic',
 'endpoint': 'rag',
 'error': None,
 'filter_accuracy': None,
 'has_results': True,
 'id': 'q001',
 'keyword

---
## Step 5 — Call `/ask` for a single query

The `/ask` endpoint does **Agentic RAG**:
1. Gemini receives the query + a tool schema (`find_rentals`)
2. Gemini calls the tool with structured arguments (price, room_type, neighbourhood…)
3. The backend runs the filtered VS2.0 search
4. Gemini receives results and generates a final answer

The key difference from `/rag`: Gemini explicitly extracts structured filters via
**function calling** — the `tool_calls` field in the response records exactly what
Gemini decided to search for.

In [23]:
# ---------------------------------------------------------------------------
# Pick q016 — multi_filter category (4 simultaneous filters)
# This is the hardest query for the agent
# ---------------------------------------------------------------------------
q016 = next(q for q in queries if q["id"] == "q016")
print(f"Query             : {q016['query']}")
print(f"Expected filters  : {q016['expected_filters']}")
print(f"Expected keywords : {q016['expected_answer_keywords']}")

Query             : 2-bedroom entire home under $120 near downtown Austin
Expected filters  : {'room_type': 'Entire home/apt', 'max_price': 120, 'min_bedrooms': 2, 'neighbourhood': 'downtown'}
Expected keywords : ['bedroom', '$120', 'downtown']


In [24]:
# ---------------------------------------------------------------------------
# Call POST /ask
#
# Request body:
#   query : str  — the natural language query (no top_k — agent decides)
#
# Timeout is longer (120s) because /ask makes multiple Gemini calls:
#   1st call: Gemini extracts filters + calls find_rentals tool
#   2nd call: Gemini synthesises the answer from retrieved listings
# ---------------------------------------------------------------------------
t0 = time.perf_counter()

ask_response = requests.post(
    f"{API_URL}/ask",
    json={"query": q016["query"]},
    timeout=120,
)
ask_response.raise_for_status()

latency_ms_ask = (time.perf_counter() - t0) * 1000
ask_data = ask_response.json()

print(f"HTTP status : {ask_response.status_code}")
print(f"Latency     : {latency_ms_ask:.0f} ms")
print(f"\nResponse keys: {list(ask_data.keys())}")

HTTP status : 200
Latency     : 110 ms

Response keys: ['answer', 'tool_calls', 'model', 'collection']


In [25]:
# ---------------------------------------------------------------------------
# Inspect the /ask response structure
#
# 'answer'     — final Gemini answer (same as /rag)
# 'tool_calls' — list of function calls Gemini made during the agentic loop
#                Each call has: { "name": "find_rentals", "args": {...}, "results": [...] }
# ---------------------------------------------------------------------------
ask_answer     = ask_data.get("answer", "")
tool_calls     = ask_data.get("tool_calls", [])

print(f"Number of tool calls: {len(tool_calls)}")
print(f"\n--- Tool calls (what Gemini chose to search for) ---")
for i, tc in enumerate(tool_calls, 1):
    print(f"\nCall #{i}:")
    print(f"  Function name : {tc.get('name')}")
    print(f"  Arguments     : {tc.get('args')}")
    results_preview = tc.get("results", [])[:2]   # show first 2 listings
    print(f"  Results count : {len(tc.get('results', []))}")
    if results_preview:
        print(f"  First result  :")
        pprint(results_preview[0])

print(f"\n--- Gemini Answer ---")
print(ask_answer)

Number of tool calls: 1

--- Tool calls (what Gemini chose to search for) ---

Call #1:
  Function name : None
  Arguments     : {'room_type': 'Entire home/apt', 'query': 'entire home near downtown Austin', 'max_price': 120, 'min_bedrooms': 2}
  Results count : 0

--- Gemini Answer ---
I found several great options for you! Here are the top 3 that best match your request for a 2-bedroom entire home under $120 near downtown Austin:

1.  **Entire 2 floor condo @ heart of ATX**
    *   **Location:** 78704 (right next to downtown!)
    *   **Price:** $105/night
    *   **Size:** 2 bedrooms, accommodates 6 guests
    *   **Rating:** 4.83 stars
    *   **Instant Bookable:** Yes
    *   **Why it's a great match:** This condo is perfectly located "right next to downtown" and offers the exact number of bedrooms you requested at a fantastic price.
    *   **Link:** https://www.airbnb.com/rooms/52454570

2.  **Riverside meets Downtown - Private Modern House**
    *   **Location:** 78741 (Riversid

---
## Step 6 — Score `/ask` with `filter_accuracy`

**`filter_accuracy`** measures how well Gemini extracted the structured filters from the query.

Formula:  
```
filter_accuracy = (number of expected filters correctly extracted) / (total expected filters)
```

Matching rules vary by filter type:
- **Numeric** (`max_price`, `min_bedrooms`, `accommodates`): within 20% tolerance
- **Boolean** (`instant_bookable`): exact match
- **Room type**: normalised fuzzy match (`"entire apt"` → `"entire home/apt"`)
- **Neighbourhood**: substring match (`"downtown"` found in `"downtown Austin"`)

In [26]:
# ---------------------------------------------------------------------------
# Source code of filter_accuracy (from eval/metrics.py)
# ---------------------------------------------------------------------------
print(inspect.getsource(filter_accuracy))

def filter_accuracy(tool_calls: list[dict], expected_filters: dict) -> float:
    """
    Measure how accurately /ask extracted the expected filters.

    Inspects tool_calls (list of {"name": "find_rentals", "args": {...}})
    and checks that each expected filter appears in at least one call.

    Returns:
        1.0  if all expected filters were correctly extracted
        0.0  if no tool calls were made (or expected_filters is empty → 1.0)
        0-1  partial credit proportional to filters matched
    """
    if not expected_filters:
        return 1.0      # no filter expectations → auto-pass

    if not tool_calls:
        return 0.0      # filters expected but agent made no tool calls

    # Collect all args from every find_rentals call
    all_args: dict[str, Any] = {}
    for call in tool_calls:
        if call.get("name") == "find_rentals":
            all_args.update(call.get("args", {}))

    matched = 0
    for key, expected_val in expected_filters.items():
        extra

In [27]:
# ---------------------------------------------------------------------------
# Step-by-step filter_accuracy for q016
#
# Expected filters: { "room_type": "Entire home/apt", "max_price": 120,
#                     "min_bedrooms": 2, "neighbourhood": "downtown" }
# ---------------------------------------------------------------------------
expected_filters = q016["expected_filters"]

# Collect all args from all find_rentals calls
all_args = {}
for call in tool_calls:
    if call.get("name") == "find_rentals":
        all_args.update(call.get("args", {}))

print(f"Expected filters : {expected_filters}")
print(f"Extracted args   : {all_args}")
print()

# Manual check per filter
print("Filter-by-filter match:")
for key, expected_val in expected_filters.items():
    extracted_val = all_args.get(key)
    if extracted_val is None:
        print(f"  {key:20s}: MISSING in tool call")
    elif isinstance(expected_val, (int, float)):
        tol = abs(float(expected_val)) * 0.20
        match = abs(float(extracted_val) - float(expected_val)) <= tol
        print(f"  {key:20s}: expected={expected_val}, extracted={extracted_val}, "
              f"within 20% → {'✓' if match else '✗'}")
    elif isinstance(expected_val, str):
        if key == "room_type":
            from eval.metrics import _normalise_room_type
            norm_exp = _normalise_room_type(expected_val)
            norm_ext = _normalise_room_type(str(extracted_val))
            match = norm_exp == norm_ext
            print(f"  {key:20s}: '{expected_val}' → normalised '{norm_exp}' "
                  f"vs '{norm_ext}' → {'✓' if match else '✗'}")
        elif key == "neighbourhood":
            match = expected_val.lower() in str(extracted_val).lower()
            print(f"  {key:20s}: '{expected_val}' in '{extracted_val}' → {'✓' if match else '✗'}")

# Use the actual metric function
fa = filter_accuracy(tool_calls, expected_filters)
print(f"\nfilter_accuracy = {fa:.4f}  ({fa*100:.0f}% of filters correctly extracted)")

Expected filters : {'room_type': 'Entire home/apt', 'max_price': 120, 'min_bedrooms': 2, 'neighbourhood': 'downtown'}
Extracted args   : {}

Filter-by-filter match:
  room_type           : MISSING in tool call
  max_price           : MISSING in tool call
  min_bedrooms        : MISSING in tool call
  neighbourhood       : MISSING in tool call

filter_accuracy = 0.0000  (0% of filters correctly extracted)


---
## Step 7 — LLM-as-judge: use Gemini to score answer quality

Keyword hit rate is mechanical — it only checks word presence, not meaning.
The **LLM-as-judge** approach uses Gemini itself to rate the answer on a 0–4 rubric:

| Score | Meaning |
|-------|---------|
| 4 | Fully answers the query — all key details, no hallucination |
| 3 | Mostly answers — minor omissions or vague details |
| 2 | Partially answers — some relevant info but significant gaps |
| 1 | Barely relevant — mostly off-topic |
| 0 | Completely irrelevant, refused, or hallucinated |

The judge model is initialised with `temperature=0.0` (deterministic) and
`max_output_tokens=8` (only needs a single digit).

In [28]:
# ---------------------------------------------------------------------------
# Look at the judge prompt template (from eval/metrics.py)
# ---------------------------------------------------------------------------
from eval.metrics import JUDGE_PROMPT
print(JUDGE_PROMPT)

You are an expert evaluator for an Airbnb listing search assistant in Austin, Texas.

Rate the ANSWER on a scale of 0 to 4 using this rubric:
  4 — Fully answers the query; all key details present; no hallucination
  3 — Mostly answers; minor omissions or vague details
  2 — Partially answers; some relevant info but significant gaps
  1 — Barely relevant; mostly off-topic or unhelpful
  0 — Completely irrelevant, refused to answer, or obvious hallucination

USER QUERY:
{query}

ANSWER:
{answer}

Respond with ONLY a single integer (0, 1, 2, 3, or 4). No explanation.



In [29]:
answer

'Here are some cozy places to relax with a great view:\n\n*   **Cozy cottage in the trees**\n    Location: 78733\n    Price: $174/night\n    Size: Accommodates 2 guests, 1 bedroom\n    URL: https://www.airbnb.com/rooms/16644078\n\n*   **Tranquil Nest With A View**\n    Location: 78741\n    Price: $81/night\n    Size: Accommodates 4 guests, 2 bedrooms\n    URL: https://www.airbnb.com/rooms/51169203\n\n*   **Remodeled Top Floor Condo w/Great Views**\n    Location: 78730\n    Price: $86/night\n    Size: Accommodates 2 guests, 2 bedrooms\n    Rating: 4.91 ★\n    URL: https://www.airbnb.com/rooms/13436718'

In [30]:
# ---------------------------------------------------------------------------
# Build the filled-in prompt for q001
# This is exactly what gets sent to Gemini as the judge
# ---------------------------------------------------------------------------
filled_prompt = build_judge_prompt(q001["query"], answer)
print("Filled judge prompt:")
print("-" * 60)
print(filled_prompt)

Filled judge prompt:
------------------------------------------------------------
You are an expert evaluator for an Airbnb listing search assistant in Austin, Texas.

Rate the ANSWER on a scale of 0 to 4 using this rubric:
  4 — Fully answers the query; all key details present; no hallucination
  3 — Mostly answers; minor omissions or vague details
  2 — Partially answers; some relevant info but significant gaps
  1 — Barely relevant; mostly off-topic or unhelpful
  0 — Completely irrelevant, refused to answer, or obvious hallucination

USER QUERY:
cozy place to relax with a great view

ANSWER:
Here are some cozy places to relax with a great view:

*   **Cozy cottage in the trees**
    Location: 78733
    Price: $174/night
    Size: Accommodates 2 guests, 1 bedroom
    URL: https://www.airbnb.com/rooms/16644078

*   **Tranquil Nest With A View**
    Location: 78741
    Price: $81/night
    Size: Accommodates 4 guests, 2 bedrooms
    URL: https://www.airbnb.com/rooms/51169203

*   **Re

In [31]:
# ---------------------------------------------------------------------------
# Initialise the Gemini judge model
#
# This uses the same project + model as the app (from config.py):
#   config.PROJECT_ID  = "deve-487713"
#   config.LOCATION    = "us-central1"
#   config.GEMINI_MODEL = "gemini-2.5-flash"
#
# temperature=0.0   → deterministic output (same score for same input)
# max_output_tokens=8 → we only need a single digit "3" or "4"
# ---------------------------------------------------------------------------
import vertexai
from vertexai.generative_models import GenerationConfig, GenerativeModel

vertexai.init(project=config.PROJECT_ID, location=config.LOCATION)

judge_model = GenerativeModel(
    model_name=config.GEMINI_MODEL,
    generation_config=GenerationConfig(
        temperature=0.0,
        max_output_tokens=64,
    ),
)
print(f"Judge model ready: {config.GEMINI_MODEL}")

Judge model ready: gemini-2.5-flash


In [32]:
judge_resp = judge_model.generate_content(filled_prompt)
print(judge_resp)

candidates {
  content {
    role: "model"
    parts {
      text: "4"
    }
  }
  finish_reason: STOP
  avg_logprobs: -9.7114572525024414
}
usage_metadata {
  prompt_token_count: 375
  candidates_token_count: 1
  total_token_count: 433
  prompt_tokens_details {
    modality: TEXT
    token_count: 375
  }
  candidates_tokens_details {
    modality: TEXT
    token_count: 1
  }
}
model_version: "gemini-2.5-flash"



In [33]:
# ---------------------------------------------------------------------------
# Call the judge model for q001
# ---------------------------------------------------------------------------
judge_resp = judge_model.generate_content(filled_prompt)
raw_score  = judge_resp.candidates[0].content.parts[0].text

print(f"Gemini raw output  : '{raw_score}'")

# parse_judge_score extracts the integer using regex: r'\b([0-4])\b'
score = parse_judge_score(raw_score)
print(f"Parsed score       : {score}")

# Explain the score
rubric = {
    4: "Fully answers — all key details, no hallucination",
    3: "Mostly answers — minor omissions",
    2: "Partially answers — significant gaps",
    1: "Barely relevant",
    0: "Irrelevant or hallucinated",
}
print(f"Score meaning      : {rubric.get(score, 'unknown')}")

Gemini raw output  : '4'
Parsed score       : 4
Score meaning      : Fully answers — all key details, no hallucination


In [34]:
# ---------------------------------------------------------------------------
# parse_judge_score handles edge cases:
#   - Model says "Score: 3" → extracts 3
#   - Model says "4\n"      → extracts 4
#   - Model says "great!"   → returns None (regex finds no digit 0-4)
# ---------------------------------------------------------------------------
print(inspect.getsource(parse_judge_score))

def parse_judge_score(raw: str) -> int | None:
    """Extract the integer score from the judge LLM's response."""
    m = re.search(r"\b([0-4])\b", raw.strip())
    return int(m.group(1)) if m else None



In [35]:
# ---------------------------------------------------------------------------
# Update rag_result_q001 with the judge score
# ---------------------------------------------------------------------------
rag_result_q001["answer_relevance"] = score
print("Updated result dict for q001:")
pprint(rag_result_q001)

Updated result dict for q001:
{'answer': 'Here are some cozy places to relax with a great view:\n'
           '\n'
           '*   **Cozy cottage in the trees**\n'
           '    Location: 78733\n'
           '    Price: $174/night\n'
           '    Size: Accommodates 2 guests, 1 bedroom\n'
           '    URL: https://www.airbnb.com/rooms/16644078\n'
           '\n'
           '*   **Tranquil Nest With A View**\n'
           '    Location: 78741\n'
           '    Price: $81/night\n'
           '    Size: Accommodates 4 guests, 2 bedrooms\n'
           '    URL: https://www.airbnb.com/rooms/51169203\n'
           '\n'
           '*   **Remodeled Top Floor Condo w/Great Views**\n'
           '    Location: 78730\n'
           '    Price: $86/night\n'
           '    Size: Accommodates',
 'answer_relevance': 4,
 'category': 'semantic',
 'endpoint': 'rag',
 'error': None,
 'filter_accuracy': None,
 'has_results': True,
 'id': 'q001',
 'keyword_hit_rate': 0.75,
 'latency_ms': 133.1,
 'q

---
## Step 8 — Run a mini evaluation (5 queries, one per category)

Now we run the full `evaluate_rag_query` / `evaluate_ask_query` functions from
`eval/evaluate.py` on a representative subset of 5 queries — one per category.

This is exactly what happens when you run:
```bash
python eval/evaluate.py --api-url $API_URL --ids q001,q006,q011,q016,q021 --no-judge
```

In [36]:
# ---------------------------------------------------------------------------
# Import the per-query evaluation functions from evaluate.py
# ---------------------------------------------------------------------------
from eval.evaluate import evaluate_rag_query, evaluate_ask_query, call_rag, call_ask

print(inspect.getsource(evaluate_rag_query))

def evaluate_rag_query(qdef: dict, api_url: str, use_judge: bool) -> dict:
    """Run one query through /rag and score it."""
    resp, latency = call_rag(qdef["query"], api_url)

    answer   = resp.get("answer", "")
    sources  = resp.get("sources", [])
    error    = resp.get("error")

    khr    = keyword_hit_rate(answer, qdef.get("expected_answer_keywords", []))
    judge  = llm_judge(qdef["query"], answer) if use_judge and answer else None

    return {
        "id":                 qdef["id"],
        "category":           qdef["category"],
        "query":              qdef["query"],
        "endpoint":           "rag",
        "answer":             answer[:500],
        "error":              error,
        "has_results":        len(sources) >= qdef.get("min_results_expected", 1),
        "retrieved_count":    len(sources),
        "latency_ms":         latency,
        "keyword_hit_rate":   khr,
        "answer_relevance":   judge,
        "filter_accuracy":    None,     # N/

In [37]:
# ---------------------------------------------------------------------------
# Smoke-test subset — one query per category
#   q001 semantic    → /rag + /ask
#   q006 price       → /ask only
#   q011 room_type   → /ask only
#   q016 multi_filter→ /ask only
#   q024 neighbourhood→ /rag + /ask
# ---------------------------------------------------------------------------
SMOKE_IDS  = ["q001", "q006", "q011", "q016", "q024"]
USE_JUDGE  = True   # set False to skip Gemini judge calls (faster, cheaper)

smoke_queries = [q for q in queries if q["id"] in SMOKE_IDS]
mini_results  = []

print(f"Running {len(smoke_queries)} queries against {API_URL}")
print(f"LLM judge: {'ON' if USE_JUDGE else 'OFF'}")
print()

for qdef in smoke_queries:
    ep    = qdef.get("endpoint", "both")
    print(f"[{qdef['id']}] ({qdef['category']}) — {qdef['query'][:55]}")

    # Decide which endpoints to call based on the 'endpoint' field
    run_rag = ep in ("rag", "both")
    run_ask = ep in ("ask", "both")

    if run_rag:
        result = evaluate_rag_query(qdef, API_URL, use_judge=USE_JUDGE)
        mini_results.append(result)
        print(f"  /rag → latency={result['latency_ms']:.0f}ms  "
              f"results={result['retrieved_count']}  "
              f"khr={result['keyword_hit_rate']:.2f}  "
              f"judge={result['answer_relevance']}")

    if run_ask:
        result = evaluate_ask_query(qdef, API_URL, use_judge=USE_JUDGE)
        mini_results.append(result)
        print(f"  /ask → latency={result['latency_ms']:.0f}ms  "
              f"filter_acc={result['filter_accuracy']:.2f}  "
              f"khr={result['keyword_hit_rate']:.2f}  "
              f"judge={result['answer_relevance']}")
    print()

print(f"Total results collected: {len(mini_results)}")

Running 5 queries against https://airbnb-rag-api-zdpzfpmtka-uc.a.run.app
LLM judge: ON

[q001] (semantic) — cozy place to relax with a great view
  /rag → latency=128ms  results=10  khr=0.75  judge=4
    [Judge] Error: Cannot get the response text.
Cannot get the Candidate text.
Response candidate content has no parts (and thus no text). The candidate is likely blocked by the safety filters.
Content:
{
  "role": "model"
}
Candidate:
{
  "content": {
    "role": "model"
  },
  "finish_reason": "MAX_TOKENS"
}
Response:
{
  "candidates": [
    {
      "content": {
        "role": "model"
      },
      "finish_reason": "MAX_TOKENS"
    }
  ],
  "usage_metadata": {
    "prompt_token_count": 1219,
    "total_token_count": 1279,
    "prompt_tokens_details": [
      {
        "modality": "TEXT",
        "token_count": 1219
      }
    ]
  },
  "model_version": "gemini-2.5-flash"
}
  /ask → latency=124ms  filter_acc=1.00  khr=1.00  judge=None

[q006] (price) — find me a private room under $60 

---
## Step 9 — Aggregate results into a summary

The `aggregate()` function in `eval/metrics.py` computes:
- Overall averages for each metric (relevance, latency, filter accuracy, khr)
- Per-category breakdown (so you can see which query type performs worst)
- Separate stats for `/rag` and `/ask` rows

In [38]:
# ---------------------------------------------------------------------------
# Source code of aggregate() for reference
# ---------------------------------------------------------------------------
print(inspect.getsource(aggregate))

def aggregate(results: list[dict]) -> dict:
    """
    Compute aggregate statistics over all eval results.

    Args:
        results: list of per-query result dicts (from evaluate.py)

    Returns:
        dict with avg/min/max for each metric, plus per-category breakdown
    """
    def safe_avg(vals):
        vs = [v for v in vals if v is not None]
        return round(sum(vs) / len(vs), 4) if vs else None

    def safe_pct(vals):
        vs = [v for v in vals if v is not None]
        return round(sum(1 for v in vs if v) / len(vs), 4) if vs else None

    rag_rows = [r for r in results if r.get("endpoint") == "rag"]
    ask_rows = [r for r in results if r.get("endpoint") == "ask"]

    def stats_for(rows, tag):
        if not rows:
            return {}
        return {
            f"{tag}_count":               len(rows),
            f"{tag}_avg_relevance":       safe_avg([r.get("answer_relevance") for r in rows]),
            f"{tag}_avg_keyword_hit_rate":safe_avg([r.get("keyword

In [39]:
# ---------------------------------------------------------------------------
# Compute summary stats for the mini evaluation
# ---------------------------------------------------------------------------
summary = aggregate(mini_results)

print("=" * 50)
print("  SUMMARY — Mini Evaluation (5 queries)")
print("=" * 50)
print(f"  Total result rows  : {summary['total_queries']}")

if "rag_count" in summary:
    print(f"\n  /rag ({summary['rag_count']} rows):")
    print(f"    Avg relevance  : {summary.get('rag_avg_relevance', 'N/A')} / 4")
    print(f"    Avg KHR        : {summary.get('rag_avg_keyword_hit_rate', 'N/A')}")
    print(f"    Has results %  : {(summary.get('rag_has_results_pct') or 0)*100:.0f}%")
    print(f"    Avg latency    : {summary.get('rag_avg_latency_ms', 'N/A'):.0f} ms")

if "ask_count" in summary:
    print(f"\n  /ask ({summary['ask_count']} rows):")
    print(f"    Avg relevance  : {summary.get('ask_avg_relevance', 'N/A')} / 4")
    print(f"    Filter accuracy: {summary.get('ask_avg_filter_accuracy', 'N/A')}")
    print(f"    Avg KHR        : {summary.get('ask_avg_keyword_hit_rate', 'N/A')}")
    print(f"    Avg latency    : {summary.get('ask_avg_latency_ms', 'N/A'):.0f} ms")

print(f"\n  By category:")
for cat, cstats in summary.get("by_category", {}).items():
    print(f"    {cat:15s}: n={cstats['count']}  "
          f"rel={cstats.get('avg_relevance', '—')}  "
          f"lat={cstats.get('avg_latency_ms', 0):.0f}ms  "
          f"has_results={cstats.get('has_results_pct', 0)*100:.0f}%")

  SUMMARY — Mini Evaluation (5 queries)
  Total result rows  : 7

  /rag (2 rows):
    Avg relevance  : 4.0 / 4
    Avg KHR        : 0.875
    Has results %  : 100%
    Avg latency    : 129 ms

  /ask (5 rows):
    Avg relevance  : 3.5 / 4
    Filter accuracy: 0.2
    Avg KHR        : 1.0
    Avg latency    : 136 ms

  By category:
    multi_filter   : n=1  rel=3.0  lat=165ms  has_results=0%
    neighbourhood  : n=2  rel=None  lat=133ms  has_results=100%
    price          : n=1  rel=4.0  lat=135ms  has_results=0%
    room_type      : n=1  rel=None  lat=123ms  has_results=0%
    semantic       : n=2  rel=4.0  lat=126ms  has_results=100%


---
## Step 10 — Visualise results in a DataFrame

Pandas makes it easy to inspect all results side-by-side, filter by category,
and spot which queries underperform.

In [40]:
import pandas as pd

# ---------------------------------------------------------------------------
# Build a DataFrame from the raw results list
# Drop the verbose 'answer' and 'tool_calls' columns for a clean view
# ---------------------------------------------------------------------------
df = pd.DataFrame(mini_results)

# Select the columns we care about
display_cols = [
    "id", "category", "endpoint",
    "has_results", "retrieved_count", "latency_ms",
    "keyword_hit_rate", "filter_accuracy", "answer_relevance",
]
df_display = df[display_cols].copy()

# Round floats for readability
df_display["keyword_hit_rate"]  = df_display["keyword_hit_rate"].round(2)
df_display["filter_accuracy"]   = df_display["filter_accuracy"].round(2)
df_display["latency_ms"]        = df_display["latency_ms"].round(0)

df_display

,id,category,endpoint,has_results,retrieved_count,latency_ms,keyword_hit_rate,filter_accuracy,answer_relevance
0,q001,semantic,rag,True,10,128.0,0.75,NaN,4.0
1,q001,semantic,ask,True,1,124.0,1.00,1.0,NaN
2,q006,price,ask,False,0,135.0,1.00,0.0,4.0
3,q011,room_type,ask,False,0,123.0,1.00,0.0,NaN
4,q016,multi_filter,ask,False,0,165.0,1.00,0.0,3.0
5,q024,neighbourhood,rag,True,10,131.0,1.00,NaN,NaN
6,q024,neighbourhood,ask,True,1,135.0,1.00,0.0,NaN


In [41]:
# ---------------------------------------------------------------------------
# Filter: show only /ask rows to inspect filter accuracy per query
# ---------------------------------------------------------------------------
ask_df = df_display[df_display["endpoint"] == "ask"].copy()
print("ASK endpoint results:")
ask_df

ASK endpoint results:


,id,category,endpoint,has_results,retrieved_count,latency_ms,keyword_hit_rate,filter_accuracy,answer_relevance
1,q001,semantic,ask,True,1,124.0,1.0,1.0,NaN
2,q006,price,ask,False,0,135.0,1.0,0.0,4.0
3,q011,room_type,ask,False,0,123.0,1.0,0.0,NaN
4,q016,multi_filter,ask,False,0,165.0,1.0,0.0,3.0
6,q024,neighbourhood,ask,True,1,135.0,1.0,0.0,NaN


In [42]:
# ---------------------------------------------------------------------------
# Show the actual answer text for any query to understand what Gemini said
# ---------------------------------------------------------------------------
def show_answer(query_id: str, endpoint: str = None):
    matches = [r for r in mini_results
               if r["id"] == query_id and (endpoint is None or r["endpoint"] == endpoint)]
    if not matches:
        print(f"No result found for {query_id} endpoint={endpoint}")
        return
    r = matches[0]
    print(f"Query [{r['id']}] /{r['endpoint']} — {r['query']}")
    print(f"Relevance score : {r['answer_relevance']} / 4")
    print(f"Keyword hit rate: {r['keyword_hit_rate']:.2f}")
    print(f"Filter accuracy : {r['filter_accuracy']}")
    print(f"Latency         : {r['latency_ms']:.0f} ms")
    print(f"\nAnswer:\n{r['answer']}")

# Show the answer for q016 /ask
show_answer("q016", "ask")

Query [q016] /ask — 2-bedroom entire home under $120 near downtown Austin
Relevance score : 3 / 4
Keyword hit rate: 1.00
Filter accuracy : 0.0
Latency         : 165 ms

Answer:
I found several great options for you! Here are the top 3 that best match your request for a 2-bedroom entire home under $120 near downtown Austin:

1.  **Entire 2 floor condo @ heart of ATX**
    *   **Location:** 78704 (right next to downtown!)
    *   **Price:** $105/night
    *   **Size:** 2 bedrooms, accommodates 6 guests
    *   **Rating:** 4.83 stars
    *   **Instant Bookable:** Yes
    *   **Why it's a great match:** This condo is perfectly located "right next to downtown" and offers th


---
## Step 11 — Build the HTML report

`eval/report.py` generates a self-contained HTML page from the results.
No external CSS or JS files — everything is inline.
The HTML is returned as a **string** (never written to disk locally) and
uploaded to GCS in Step 12.

In [43]:
# ---------------------------------------------------------------------------
# Build the HTML report as a string
# Input:
#   results — list of per-query result dicts (same structure we built above)
#   summary — aggregate() output
# Output: HTML string (~30-50KB self-contained page)
# ---------------------------------------------------------------------------
html_report = build_html_report(mini_results, summary)

print(f"Report type      : {type(html_report)}")
print(f"Report length    : {len(html_report):,} characters")
print(f"Report encoding  : {len(html_report.encode('utf-8')):,} bytes")
print()
print("First 500 characters of report:")
print(html_report[:500])

Report type      : <class 'str'>
Report length    : 21,032 characters
Report encoding  : 21,109 bytes

First 500 characters of report:
<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width,initial-scale=1">
<title>RAG Evaluation Report — 2026-03-21 12:20 UTC</title>
<style>
  * { box-sizing: border-box; margin: 0; padding: 0; }
  body { font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", sans-serif;
          background: #F8FAFC; color: #1E293B; padding: 24px; }
  h1   { font-size: 22px; font-weight: 700; margin-bottom: 4px; }
  h2   { font-size: 16px; font-weig


In [44]:
# ---------------------------------------------------------------------------
# Preview the report inside the notebook (rendered HTML)
# This is for inspection only — the actual file goes to GCS in Step 12
# ---------------------------------------------------------------------------
from IPython.display import HTML, display

# Wrap in a scrollable iframe-like container so it doesn't overflow the notebook
preview = f"""
<div style="border:2px solid #E2E8F0; border-radius:8px; padding:4px;
            height:500px; overflow:auto; background:#F8FAFC">
{html_report}
</div>
"""
display(HTML(preview))

---
## Step 12 — Upload results to GCS

The actual `eval/evaluate.py` script uploads 4 files per run:

| File | Purpose |
|------|---------|
| `eval/eval_YYYYMMDD_HHMMSS.json` | Timestamped archive — never overwritten |
| `eval/eval_YYYYMMDD_HHMMSS.html` | Timestamped archive — never overwritten |
| `eval/latest.json` | Always the most recent run |
| `eval/latest.html` | Always the most recent run |

Bucket and prefix come from `config.py` — never hardcoded.

In [45]:
# ---------------------------------------------------------------------------
# Inspect the upload logic from evaluate.py
# ---------------------------------------------------------------------------
from eval.evaluate import save_results_to_gcs, _upload_bytes
print(inspect.getsource(_upload_bytes))

def _upload_bytes(bucket_name: str, gcs_path: str, data: bytes, content_type: str) -> str:
    """Upload bytes to GCS and return the public gs:// URI."""
    storage_client = gcs.Client(project=config.PROJECT_ID)
    bucket = storage_client.bucket(bucket_name)
    blob   = bucket.blob(gcs_path)
    blob.upload_from_string(data, content_type=content_type)
    return f"gs://{bucket_name}/{gcs_path}"



In [46]:
print(inspect.getsource(save_results_to_gcs))

def save_results_to_gcs(results: list[dict], summary: dict) -> dict[str, str]:
    """
    Upload eval results (JSON + HTML) to GCS.

    Writes two sets of files:
      timestamped  → eval/eval_YYYYMMDD_HHMMSS.{json,html}
      latest       → eval/latest.{json,html}  (overwritten each run)

    Returns dict of {json_uri, html_uri, latest_json_uri, latest_html_uri}
    """
    ts     = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
    prefix = getattr(config, "EVAL_GCS_PREFIX", "eval/")
    bucket = config.BUCKET_NAME

    payload = {"summary": summary, "results": results}

    # ── JSON ──────────────────────────────────────────────────────────────────
    json_bytes = json.dumps(payload, indent=2, ensure_ascii=False).encode()
    ts_json    = _upload_bytes(bucket, f"{prefix}eval_{ts}.json",
                               json_bytes, "application/json")
    latest_json = _upload_bytes(bucket, f"{prefix}latest.json",
                                json_bytes, "application/json"

In [47]:
# ---------------------------------------------------------------------------
# Upload the mini evaluation results to GCS
#
# GCS paths from config.py:
#   config.BUCKET_NAME    = "rag-airbnb-listing"
#   config.EVAL_GCS_PREFIX = "eval/"
#
# Files written:
#   gs://rag-airbnb-listing/eval/eval_YYYYMMDD_HHMMSS.json
#   gs://rag-airbnb-listing/eval/eval_YYYYMMDD_HHMMSS.html
#   gs://rag-airbnb-listing/eval/latest.json
#   gs://rag-airbnb-listing/eval/latest.html
# ---------------------------------------------------------------------------
print(f"Uploading to: gs://{config.BUCKET_NAME}/{config.EVAL_GCS_PREFIX}")

uris = save_results_to_gcs(mini_results, summary)

print("\n✓  Uploaded:")
for label, uri in uris.items():
    print(f"   {label:20s}: {uri}")

Uploading to: gs://rag-airbnb-listing/eval/

✓  Uploaded:
   json_uri            : gs://rag-airbnb-listing/eval/eval_20260321_122039.json
   html_uri            : gs://rag-airbnb-listing/eval/eval_20260321_122039.html
   latest_json_uri     : gs://rag-airbnb-listing/eval/latest.json
   latest_html_uri     : gs://rag-airbnb-listing/eval/latest.html


In [48]:
# ---------------------------------------------------------------------------
# Download the HTML report from GCS and open it locally
# ---------------------------------------------------------------------------
import subprocess

local_path = "/tmp/eval_notebook_report.html"
latest_uri = uris["latest_html_uri"]

subprocess.run(["gsutil", "cp", latest_uri, local_path], check=True)
print(f"Downloaded to: {local_path}")
print(f"Open with:")
print(f"  open {local_path}")
print(f"  OR: from IPython.display import HTML; HTML(open('{local_path}').read())")

Copying gs://rag-airbnb-listing/eval/latest.html...
/ [1 files][ 20.6 KiB/ 20.6 KiB]                                                
Operation completed over 1 objects/20.6 KiB.                                     


Downloaded to: /tmp/eval_notebook_report.html
Open with:
  open /tmp/eval_notebook_report.html
  OR: from IPython.display import HTML; HTML(open('/tmp/eval_notebook_report.html').read())


---
## Step 13 — Bonus: Running the full evaluation in one call

Now that you understand every piece, here is how to run the full 25-query
evaluation using the same `run_evaluation()` function that `evaluate.py` calls.

**Expected duration**: 3–8 minutes (25 queries × ~2 Gemini calls each)  
**Second run** (same queries): < 30 seconds (Redis cache HITs for all)

In [49]:
# ---------------------------------------------------------------------------
# Full evaluation — all 25 queries, both endpoints, LLM judge ON
#
# This is equivalent to:
#   python eval/evaluate.py --api-url $API_URL
#
# Uncomment and run when ready (takes 3-8 minutes on first run)
# ---------------------------------------------------------------------------
from eval.evaluate import run_evaluation

# Uncomment to run:
# t0 = time.time()
# all_results = run_evaluation(
#     api_url   = API_URL,
#     endpoint  = None,      # None = both /rag and /ask
#     ids       = None,      # None = all 25 queries
#     use_judge = True,      # set False to skip Gemini judge (faster)
# )
# elapsed = time.time() - t0
# full_summary = aggregate(all_results)
# print(f"\nDone in {elapsed:.0f}s — {len(all_results)} result rows")
# pprint(full_summary)

print("Uncomment the lines above and run this cell to execute the full evaluation.")

Uncomment the lines above and run this cell to execute the full evaluation.


In [50]:
# ---------------------------------------------------------------------------
# Quick smoke test — 5 queries, no judge (fastest option, no Gemini cost)
#
# Equivalent to:
#   python eval/evaluate.py --api-url $API_URL \
#     --ids q001,q006,q011,q016,q021 --no-judge
# ---------------------------------------------------------------------------
t0 = time.time()
smoke_results = run_evaluation(
    api_url   = API_URL,
    endpoint  = None,
    ids       = ["q001", "q006", "q011", "q016", "q021"],
    use_judge = False,   # skip LLM judge — 5x faster, zero Gemini cost
)
elapsed = time.time() - t0

smoke_summary = aggregate(smoke_results)
print(f"\nSmoke test done in {elapsed:.1f}s")
print(f"Rows collected: {len(smoke_results)}")
print(f"\n/rag avg latency  : {smoke_summary.get('rag_avg_latency_ms', 'N/A')} ms")
print(f"/ask avg latency  : {smoke_summary.get('ask_avg_latency_ms', 'N/A')} ms")
print(f"/ask filter acc   : {smoke_summary.get('ask_avg_filter_accuracy', 'N/A')}")


[01/05] q001 (semantic) — "cozy place to relax with a great view"
  → /rag ... latency=122ms  results=10  khr=0.75  judge=None
  → /ask ... latency=136ms  filter_acc=1.00  khr=1.00  judge=None

[02/05] q006 (price) — "find me a private room under $60 a night"
  → /ask ... latency=115ms  filter_acc=0.00  khr=1.00  judge=None

[03/05] q011 (room_type) — "I need a private room in someone's home, not the whole place"
  → /ask ... latency=115ms  filter_acc=0.00  khr=1.00  judge=None

[04/05] q016 (multi_filter) — "2-bedroom entire home under $120 near downtown Austin"
  → /ask ... latency=120ms  filter_acc=0.00  khr=1.00  judge=None

[05/05] q021 (neighbourhood) — "anything available in East Austin"
  → /ask ... latency=4348ms  filter_acc=0.00  khr=1.00  judge=None

Smoke test done in 5.0s
Rows collected: 6

/rag avg latency  : 122.3 ms
/ask avg latency  : 966.62 ms
/ask filter acc   : 0.2


In [51]:
# ---------------------------------------------------------------------------
# Cache effect: run the same 5 queries again — should be much faster
# Results come from Redis (0.5-2ms) instead of VS2.0 + Gemini (800-3000ms)
# ---------------------------------------------------------------------------
print("Running the same 5 queries again to demonstrate Redis cache speedup...")
t0 = time.time()
cached_results = run_evaluation(
    api_url   = API_URL,
    endpoint  = None,
    ids       = ["q001", "q006", "q011", "q016", "q021"],
    use_judge = False,
)
elapsed_cached = time.time() - t0
cached_summary = aggregate(cached_results)

print(f"\nFirst run  : {elapsed:.1f}s (cache MISS — full VS2.0 + Gemini pipeline)")
print(f"Second run : {elapsed_cached:.1f}s (cache HIT — Redis returns pre-computed results)")
print(f"Speedup    : {elapsed/elapsed_cached:.1f}x faster")

# Check cache stats
cache_resp = requests.get(f"{API_URL}/cache/stats", timeout=5)
if cache_resp.ok:
    cache_stats = cache_resp.json()
    print(f"\nCache stats:")
    print(f"  Total keys : {cache_stats.get('total_keys', '?')}")
    print(f"  Hits       : {cache_stats.get('hits', '?')}")
    print(f"  Misses     : {cache_stats.get('misses', '?')}")

Running the same 5 queries again to demonstrate Redis cache speedup...

[01/05] q001 (semantic) — "cozy place to relax with a great view"
  → /rag ... latency=161ms  results=10  khr=0.75  judge=None
  → /ask ... latency=167ms  filter_acc=1.00  khr=1.00  judge=None

[02/05] q006 (price) — "find me a private room under $60 a night"
  → /ask ... latency=137ms  filter_acc=0.00  khr=1.00  judge=None

[03/05] q011 (room_type) — "I need a private room in someone's home, not the whole place"
  → /ask ... latency=138ms  filter_acc=0.00  khr=1.00  judge=None

[04/05] q016 (multi_filter) — "2-bedroom entire home under $120 near downtown Austin"
  → /ask ... latency=130ms  filter_acc=0.00  khr=1.00  judge=None

[05/05] q021 (neighbourhood) — "anything available in East Austin"
  → /ask ... latency=133ms  filter_acc=0.00  khr=1.00  judge=None

First run  : 5.0s (cache MISS — full VS2.0 + Gemini pipeline)
Second run : 0.9s (cache HIT — Redis returns pre-computed results)
Speedup    : 5.6x faster

Ca

---
## Summary: How everything connects

```
eval/dataset.json
  └── 25 labeled queries
        │
        ▼
eval/evaluate.py :: run_evaluation()
  ├── for each query:
  │     ├── call_rag()  → POST /rag  → { answer, sources }
  │     └── call_ask()  → POST /ask  → { answer, tool_calls }
  │
  ├── eval/metrics.py
  │     ├── keyword_hit_rate(answer, expected_keywords) → 0.0–1.0
  │     ├── filter_accuracy(tool_calls, expected_filters) → 0.0–1.0
  │     ├── llm_judge(query, answer) → 0–4  ← calls Gemini
  │     └── aggregate(results) → summary dict
  │
  ├── eval/report.py
  │     └── build_html_report(results, summary) → HTML string
  │
  └── save_results_to_gcs()
        ├── gs://{BUCKET}/eval/eval_YYYYMMDD.json
        ├── gs://{BUCKET}/eval/eval_YYYYMMDD.html
        ├── gs://{BUCKET}/eval/latest.json   ← overwritten each run
        └── gs://{BUCKET}/eval/latest.html   ← overwritten each run
```

### Quick reference: CLI equivalents
```bash
# Full run (25 queries, both endpoints, LLM judge)
python eval/evaluate.py --api-url $API_URL

# Smoke test (5 queries, no judge)
python eval/evaluate.py --api-url $API_URL --ids q001,q006,q011,q016,q021 --no-judge

# Only semantic RAG
python eval/evaluate.py --api-url $API_URL --endpoint rag

# Only agentic
python eval/evaluate.py --api-url $API_URL --endpoint ask

# Download and open the latest report
BUCKET=$(python3 -c "import config; print(config.BUCKET_NAME)")
PREFIX=$(python3 -c "import config; print(config.EVAL_GCS_PREFIX)")
gsutil cp gs://$BUCKET/${PREFIX}latest.html /tmp/eval.html && open /tmp/eval.html
```